In [63]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [64]:
spark = SparkSession.builder \
    .appName("ManhattanPlotDataPrep") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

In [65]:
disease_ta_path = "/Users/polina/genetics_gsea/data/genes_therapeutic_areas"
disease_ta = spark.read.parquet(disease_ta_path).dropDuplicates().drop("approvedsymbol")

In [ ]:
disease_ta.show(2, truncate=False)

+---------------+--------------+----------------------+------------+------------+------+--------------+----------------------+----------------------+-----------------------+-------------------+-----------------+---------------------------+----------------------+---------------------+---------------+-----------------------+---------------------------------+--------------------------+----------------------+----------------------------+--------------------+----------------------------------------+-------------+-------------------+------------------+--------------------+-------------------+-----------------------------+----------------------------------+----------------------------------+-------------+-----+------------+-------------+-------------+-------------+------------+-----------------+------------------+
|geneId         |uniqueDiseases|uniqueTherapeuticAreas|maxEQTLColoc|maxPQTLColoc|maxVEP|maxDistanceTSS|minEffectiveSampleSize|maxEffectiveSampleSize|earliestPublicationDate|cancerOr

In [ ]:
measur_path = "/Users/polina/genetics_gsea/data/unique_measurement_count_per_gene.parquet"
measur = spark.read.parquet(measur_path).withColumnRenamed("unique_disease_count", "uniqueMeasurement")

In [ ]:
measur.show(2, truncate=False)

+---------------+-----------------+
|geneId         |uniqueMeasurement|
+---------------+-----------------+
|ENSG00000224578|39               |
|ENSG00000115705|5                |
+---------------+-----------------+
only showing top 2 rows


In [69]:
target_index_path = "/Users/polina/genetics_gsea/data/target_index_for_plot.parquet"
target_index = spark.read.parquet(target_index_path).withColumnRenamed("targetId", "geneId")

In [70]:
target_index.show(2, truncate=False)

+---------------+----------+---------+---------+--------------+
|geneId         |chromosome|start    |end      |approvedSymbol|
+---------------+----------+---------+---------+--------------+
|ENSG00000000003|X         |100627108|100639991|TSPAN6        |
|ENSG00000000005|X         |100584936|100599885|TNMD          |
+---------------+----------+---------+---------+--------------+
only showing top 2 rows


### Merge by geneId

In [54]:
disease_ta_measur = disease_ta.join(measur, on="geneId", how="outer")

In [74]:
disease_ta_index = disease_ta.join(target_index, on="geneId", how="inner")

In [76]:
disease_ta_index.show(2, truncate=False)

+---------------+--------------+----------------------+------------+------------+------+--------------+----------------------+----------------------+-----------------------+-------------------+-----------------+---------------------------+----------------------+---------------------+---------------+-----------------------+---------------------------------+--------------------------+----------------------+----------------------------+--------------------+----------------------------------------+-------------+-------------------+------------------+--------------------+-------------------+-----------------------------+----------------------------------+----------------------------------+-------------+-----+------------+-------------+-------------+-------------+------------+-----------------+------------------+----------+---------+---------+--------------+
|geneId         |uniqueDiseases|uniqueTherapeuticAreas|maxEQTLColoc|maxPQTLColoc|maxVEP|maxDistanceTSS|minEffectiveSampleSize|maxEffect

In [75]:
disease_ta_index.filter(col("chromosome") == "Y").count()

0

In [82]:
disease_ta_index_pandas = disease_ta_index.toPandas()

In [85]:
disease_ta_index_pandas.to_csv("/Users/polina/genetics_gsea/data/disease_ta_index_pandas.csv")

In [55]:
disease_ta.count()

8285

In [84]:
disease_ta_index.filter(col("chromosome") == "22").show()

+---------------+--------------+----------------------+------------+------------+------+--------------+----------------------+----------------------+-----------------------+-------------------+-----------------+---------------------------+----------------------+---------------------+---------------+-----------------------+---------------------------------+--------------------------+----------------------+----------------------------+--------------------+----------------------------------------+-------------+-------------------+------------------+--------------------+-------------------+-----------------------------+----------------------------------+----------------------------------+-------------+-----+------------+-------------+-------------+-------------+------------+-----------------+------------------+----------+--------+--------+--------------+
|         geneId|uniqueDiseases|uniqueTherapeuticAreas|maxEQTLColoc|maxPQTLColoc|maxVEP|maxDistanceTSS|minEffectiveSampleSize|maxEffectiv

In [77]:
measur_index = measur.join(target_index, on="geneId", how="inner")

In [80]:
target_index.filter(col("chromosome") == "Y").count()

61

In [78]:
measur_index.filter(col("chromosome") == "Y").count()

0

In [86]:
measur_index.toPandas().to_csv("/Users/polina/genetics_gsea/data/measur_index_pandas.csv")

25/09/24 23:25:00 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 900073 ms exceeds timeout 120000 ms
25/09/24 23:25:00 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/24 23:42:00 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o

In [56]:
measur.count()

15160

In [57]:
disease_ta_measur.count()

15641

In [58]:
disease_ta_measur_index = disease_ta_measur.join(target_index, on="geneId", how="inner")

In [59]:
disease_ta_measur_index.show(2, truncate=False)

+---------------+--------------+----------------------+------------+------------+------+--------------+----------------------+----------------------+-----------------------+-------------------+-----------------+---------------------------+----------------------+---------------------+---------------+-----------------------+---------------------------------+--------------------------+----------------------+----------------------------+--------------------+----------------------------------------+-------------+-------------------+------------------+--------------------+-------------------+-----------------------------+----------------------------------+----------------------------------+-------------+-----+------------+-------------+-------------+-------------+------------+-----------------+------------------+-----------------+----------+---------+---------+--------------+
|geneId         |uniqueDiseases|uniqueTherapeuticAreas|maxEQTLColoc|maxPQTLColoc|maxVEP|maxDistanceTSS|minEffectiveSa

In [62]:
disease_ta_measur_index.filter(col("chromosome") == "Y").count()

0

In [81]:
disease_ta_measur_index.filter(col("chromosome") == "21").count()

154

In [61]:
disease_ta_measur_index.write.mode("overwrite").parquet("/Users/polina/genetics_gsea/data/disease_ta_measur_index")